In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import os
import logging
from sklearn.feature_selection import VarianceThreshold

from sklearn.preprocessing import StandardScaler , MinMaxScaler, RobustScaler
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
CLEANED_DATA_PATH="./../data/cleaned/cleaned_data.csv"
TRANSFORMATION_LOG_REPORT_PATH="./../reports/transformation_log_report.csv"
TRANSFORMATION_LOGGING_PATH="./../reports/transformation.log"

# Prepare for logging

In [ ]:
logging.basicConfig(
    level=logging.INFO,
    handlers=[
        logging.FileHandler(TRANSFORMATION_LOGGING_PATH),   # writes to file
        logging.StreamHandler(),               # prints to console
    ],
)

transformation_log_Report: list[dict] = []

In [ ]:
def log_cleaning_action(stage: str, column: str, action: str, reason: str) -> None:
    transformation_log_Report.append({
        "stage": stage,
        "column": column,
        "action": action,
        "reason": reason,
    })
    logging.info(f"[LOG] {stage} | {column} | {action} | {reason}")

# load data

In [ ]:
df = pd.read_csv(CLEANED_DATA_PATH)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def show_histogram_with_stats(df, column_name):
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(5, 3))
    sns.histplot(df[column_name], kde=True, color="skyblue", bins=50, alpha=0.5)
    plt.axvline(df[column_name].mean(), color="red", linestyle="dashed", linewidth=1, label="Mean")
    plt.axvline(df[column_name].median(), color="green", linestyle="dashed", linewidth=1, label="Median")
    plt.legend()
    plt.title('Original ' + column_name.capitalize() + ' Distribution')
    plt.xlabel(column_name.capitalize())
    plt.ylabel('Count')
    plt.show()


# define our target 


In [ ]:
#split inot new col for the values of price_egp
show_histogram_with_stats(df, 'price_egp')


In [ ]:
# binning the price_egp using qcut to create 3 bins (0-100, 100-200, 200+)

df['price_egp_bin'] = pd.qcut(df['price_egp'], q=3, labels=[0,1,2])

In [ ]:
df['price_egp_bin'].value_counts()

# Split Data into Train and Test Sets   

In [ ]:
# split data into train and test sets
from sklearn.model_selection import train_test_split
X = df.drop(['price_egp', 'price_egp_bin'], axis=1)
# X = df.drop('price_category', axis=1)
y = df['price_egp_bin']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


In [ ]:
# split trian to train and validation sets
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [ ]:
# ### need to move 
# X_train.loc[X_train["amenities"] == "No amenities listed", "amenities"] = ""
# X_val.loc[X_val["amenities"] == "No amenities listed", "amenities"] = ""
# X_train["amenities_count"] = (
#     X_train["amenities"]
#     .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
# )

# X_val["amenities_count"] = (
#     X_val["amenities"]
#     .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
# )
# X_test.loc[X_test["amenities"] == "No amenities listed", "amenities"] = ""
# X_test["amenities_count"] = (   
#     X_test["amenities"]
#     .apply(lambda x: 0 if x == "" else len([i.strip() for i in x.split("|")]))
# )


In [ ]:
X_train_raw=X_train.copy()  
X_val_raw=X_val.copy()
X_test_raw=X_test.copy()


In [ ]:
X_train=X_train_raw
X_val=X_val_raw
X_test=X_test_raw

# Feature Scaling 
-   standardization
-   Min-Max Scaling
-   Robust Scaling 

**Transform numerical features to a common scale.**

### lat, lon 
lat:
- mean 29.88
- std 0.66
- min 25
- max 30.99
- median 30.01

lon:
- mean 31.46
- std 0.69
- min 27.97
- max 34.89
- median 31.25

**will use StandardScaler**

In [ ]:
show_histogram_with_stats(X_train, 'lat')
show_histogram_with_stats(X_train, 'lon')

In [ ]:
# apply scaling 
scaler = RobustScaler()
X_train['lat'] = scaler.fit_transform(X_train[['lat']])
X_val['lat'] = scaler.transform(X_val[['lat']])
X_test['lat'] = scaler.transform(X_test[['lat']])
show_histogram_with_stats(X_train, 'lat')



scaler = RobustScaler()
X_train['lon'] = scaler.fit_transform(X_train[['lon']])
X_val['lon'] = scaler.transform(X_val[['lon']])
X_test['lon'] = scaler.transform(X_test[['lon']])
show_histogram_with_stats(X_train, 'lon')




###  (4) **area_value**
-   mean=145
-   median=145
-   Q1=116
-   Q3=173
-   max=765

**mean = median** <br>
**So normal distribution so will apply Standardization**

In [ ]:
show_histogram_with_stats(X_train, 'area_value')


In [ ]:

# apply scaling 
scaler = StandardScaler()
X_train['area_value'] = scaler.fit_transform(X_train[['area_value']])
X_val['area_value'] = scaler.transform(X_val[['area_value']])
X_test['area_value'] = scaler.transform(X_test[['area_value']])
#after
show_histogram_with_stats(X_train, 'area_value')



###  (6) **distance features**
- *dist_nearest_school_km*
- *dist_nearest_hospital_km*
- *dist_nearest_supermarket_km*
- *dist_nearest_mall_km*
- *dist_nearest_transit_station_km*
- *dist_nearest_cafe_restaurant_km*

right skewed <br>
**So will apply Robust Scaling**

In [ ]:
for col in X_train.columns:
    if col.startswith('dist_nearest'):
        show_histogram_with_stats(X_train, col)

In [ ]:
X_train.columns

In [ ]:
col=['dist_nearest_mall_km', 'dist_nearest_transit_station_km']

for c in col:
    scaler= StandardScaler()
    X_train[c] = scaler.fit_transform(X_train[[c]])
    X_val[c] = scaler.transform(X_val[[c]])
    X_test[c] = scaler.transform(X_test[[c]])
    show_histogram_with_stats(X_train, c)




col =[ 'dist_nearest_school_km','dist_nearest_hospital_km', 'dist_nearest_supermarket_km','dist_nearest_cafe_restaurant_km']

for c in col:
    if c in X_train.columns:
        scaler = RobustScaler()
        X_train[c] = scaler.fit_transform(X_train[[c]])
        X_val[c] = scaler.transform(X_val[[c]])
        X_test[c] = scaler.transform(X_test[[c]])
        show_histogram_with_stats(X_train, c)

###  (7) **Count features**
- *school_count_within_3km*
- *hospital_count_within_3km*
- *supermarket_count_within_3km*
- *mall_count_within_3km*
- *transit_station_count_within_3km*
- *cafe_restaurant_count_within_3km*

bounded values non negative. <br>
**So will apply min-max scaling**

In [ ]:
for col in X_train.columns:
    if col.endswith('count_within_3km'):
        show_histogram_with_stats(X_train, col)

In [ ]:
for col in X_train.columns:
    if col.endswith('count_within_3km'):
        scaler = RobustScaler()
        X_train[col] = scaler.fit_transform(X_train[[col]])
        X_val[col] = scaler.transform(X_val[[col]])
        X_test[col] = scaler.transform(X_test[[col]])
        show_histogram_with_stats(X_train, col)

In [ ]:
X_train_Scaled=X_train.copy()  
X_val_Scaled=X_val.copy()
X_test_Scaled=X_test.copy()


In [ ]:
X_train=X_train_Scaled
X_val=X_val_Scaled
X_test=X_test_Scaled

# Feature Encoding 
-   One-Hot Encoding
-   Label Encoding
-   Target Encoding
-   Binary Encoding
-   Frequency Encoding
-   Rare Encoding

### (1) **Ordinal encode**  listing_level 
-   *standard* = 0
-  *featured* = 1
-  *premium* = 2
-  *hot* = 3
-  *superhot* = 4

as there is a clear order will apply Label Encoding

In [ ]:
listing_map = {
    "standard": 0,
    "featured": 1,
    "premium": 2,
    "hot": 3,
    "superhot": 4
}

for df_ in [X_train, X_val, X_test]:
    df_["listing_level"] = df_["listing_level"].map(listing_map)

### (2) **completion_status**              
-   *under-construction*    
-   *off_plan*      
-   *completed*     


### (3) **furnished**
-   *Unfurnished*    
-   *Unknown*         
-   *Furnished*        
-   *PARTLY*

we will apply one-hot encoding for both completion_status and furnished as they are nominal categorical variables with no clear order.

In [ ]:
cat_cols = ["completion_status", "furnished"]

X_train = pd.get_dummies(X_train, columns=cat_cols, drop_first=True)
X_val = pd.get_dummies(X_val, columns=cat_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=cat_cols, drop_first=True)


X_val = X_val.reindex(columns=X_train.columns, fill_value=0)
X_test = X_test.reindex(columns=X_train.columns, fill_value=0)

### (4) Frequency encode 
-   *city*
-   *town*
-   *district*

In [ ]:
# can make with freq of city with high price
freq_cols = ["city", "town", "district"]

for col in freq_cols:
    freq_map = X_train[col].value_counts(normalize=True)

    for df in [X_train, X_val, X_test]:
        df[col] = df[col].map(freq_map).fillna(0)

        

### (5) amenities convert to binary features


In [ ]:
# # need to check
# X_train['amenities'] = X_train['amenities'].fillna('No amenities listed').str.replace(' | ', '|', regex=False)
# X_val['amenities']   = X_val['amenities'].fillna('No amenities listed').str.replace(' | ', '|', regex=False)
# X_test['amenities']  = X_test['amenities'].fillna('No amenities listed').str.replace(' | ', '|', regex=False)

# train_amenities = X_train['amenities'].str.get_dummies(sep='|')
# val_amenities   = X_val['amenities'].str.get_dummies(sep='|')
# test_amenities  = X_test['amenities'].str.get_dummies(sep='|')

# val_amenities  = val_amenities.reindex(columns=train_amenities.columns, fill_value=0)
# test_amenities = test_amenities.reindex(columns=train_amenities.columns, fill_value=0)

# X_train = pd.concat([X_train.drop(columns=['amenities']), train_amenities], axis=1)
# X_val   = pd.concat([X_val.drop(columns=['amenities']),   val_amenities],   axis=1)
# X_test  = pd.concat([X_test.drop(columns=['amenities']),  test_amenities],  axis=1)

###  **convert bool to  int**


In [ ]:
for df in [X_train, X_val, X_test]:
    bool_cols = df.select_dtypes(bool).columns
    df[bool_cols] = df[bool_cols].astype(int)

In [ ]:
len(X_train.columns.tolist())

In [ ]:
X_train_encoded = X_train.copy()
X_val_encoded = X_val.copy()
X_test_encoded = X_test.copy()


In [ ]:
X_train=X_train_encoded
X_val=X_val_encoded
X_test=X_test_encoded

# Feature Interactions
-   Arithmetic Complications
-   Statistical Aggregations
-   Boolean and Logical Combinations


In [ ]:
X_train['luxury_score']= X_train[['private pool', "private garden", 'view of water', 'covered parking']].sum(axis=1)
X_train['bedroom_density']= X_train['bedrooms'] / (X_train['area_value'] + 1e-5) 
X_train['bathroom_density']= X_train['bathrooms'] / (X_train['area_value'] + 1e-5)
X_train['total_rooms']= X_train['bedrooms'] + X_train['bathrooms']
X_train['room_density']= X_train['total_rooms'] / (X_train['area_value'] + 1e-5)

X_train['accessibility_score']= X_train['school_count_within_3km'] + X_train['hospital_count_within_3km'] + X_train['supermarket_count_within_3km']+ X_train['transit_station_count_within_3km']
X_train['lifestyle_score'] = X_train['mall_count_within_3km '] + X_train['cafe_restaurant_count_within_3km']


In [ ]:
X_train.columns

In [ ]:

X_train = X_train.drop(columns=["amenities"])
X_val = X_val.drop(columns=["amenities"])
X_test = X_test.drop(columns=["amenities"])

In [ ]:

for col in X_train.columns:
    unique_values = X_train[col].nunique()
    print(f"{col}: {unique_values} unique values")
    print(f"Values: {X_train[col].dtype}\n")

# Feature Selection
-   Filter Methods
-   Wrapper Methods
-   pemutation importance

### Step 1: Variance Threshold

In [ ]:

selector = VarianceThreshold(threshold=0.01)

X_train_var = selector.fit_transform(X_train)

selected_features = X_train.columns[selector.get_support()]

X_train = pd.DataFrame(X_train_var, columns=selected_features, index=X_train.index)
X_val = X_val[selected_features]
X_test = X_test[selected_features]

print("Remaining features:", len(selected_features))
print(selected_features)

### Step 2: Correlation with target

In [ ]:
corr = X_train.corrwith(y_train).abs().sort_values(ascending=False)
corr.head(20).plot(kind="barh")
plt.title("Top correlated features with target")
plt.show()


**Remove weak target correlation**

In [ ]:
selected_corr = corr[corr > 0.05].index.tolist()
print(selected_corr)

# X_train = X_train[selected_corr]
# X_val = X_val[selected_corr]
# X_test = X_test[selected_corr]


### Step 3: Remove multicollinearity

In [ ]:
corr_matrix = X_train.corr().abs()


upper = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)
to_drop = []

for col in upper.columns:
    high_corr_features = upper.index[upper[col] > 0.9].tolist()

    for row in high_corr_features:
        corr_row = abs(X_train[row].corr(y_train))
        corr_col = abs(X_train[col].corr(y_train))

        if corr_row >= corr_col:
            to_drop.append(col)
        else:
            to_drop.append(row)

to_drop = list(set(to_drop))
print("Drop:", to_drop)

In [ ]:


plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0)
plt.title("Correlation Heatmap")
plt.show()


In [ ]:
# X_train = X_train.drop(columns=to_drop)
# X_val = X_val.drop(columns=to_drop)
# X_test = X_test.drop(columns=to_drop)

## 2. Wrapper Methods

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rfe = RFE(model, n_features_to_select=15)
rfe.fit(X_train, y_train)

In [ ]:
selected_rfe = X_train.columns[rfe.support_]

print(selected_rfe)

In [ ]:
# X_train = X_train[selected_rfe]
# X_val = X_val[selected_rfe]
# X_test = X_test[selected_rfe]

### RFECV

In [ ]:
from sklearn.feature_selection import RFECV

rfecv = RFECV(
    estimator=model,
    step=1,
    cv=5,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1
)

rfecv.fit(X_train, y_train)

In [ ]:
selected_rfecv = X_train.columns[rfecv.support_]

print("Optimal features:", len(selected_rfecv))
print(selected_rfecv)

In [ ]:
# X_train = X_train[selected_rfecv]
# X_val = X_val[selected_rfecv]
# X_test = X_test[selected_rfecv]

### 

In [ ]:
model.fit(X_train, y_train)

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_val,
    y_val,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

In [ ]:

feature_importances = pd.DataFrame({
    "feature": X_val.columns,
    "importance": perm.importances_mean,
    "std": perm.importances_std
}).sort_values("importance", ascending=False)

print(feature_importances.head(20))

In [ ]:
important_features = feature_importances[
    feature_importances["importance"] > 0
]["feature"].tolist()

In [ ]:
# X_train = X_train[important_features]
# X_val = X_val[important_features]
# X_test = X_test[important_features]

# Documentation
